# First experiment with AstroPy


In [10]:
from datetime import datetime, timedelta, timezone
import importlib

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import earth_moon_model

importlib.reload(earth_moon_model)

import earth_moon_orbit_plot

importlib.reload(earth_moon_orbit_plot)

from earth_moon_model import moon_orbit_positions
from earth_moon_orbit_plot import (
    create_earth_moon_orbit_plot,
    plot_to_png,
    update_earth_moon_orbit_plot,
)

TRAIL_DAYS = 28
TIME_STEP_HOURS = 1
TOTAL_HOURS = 90 * 24
START_DATE = datetime(2026, 1, 1, tzinfo=timezone.utc)

orbit_plot = create_earth_moon_orbit_plot()
plt.close(orbit_plot.figure)
plot_image = widgets.Image(format="png", layout=widgets.Layout(width="720px", height="720px"))


def update_plot(change):
    current_timestamp = START_DATE + timedelta(hours=change["new"])
    trail_start = max(START_DATE, current_timestamp - timedelta(days=TRAIL_DAYS))
    orbit_samples = moon_orbit_positions(trail_start, current_timestamp)

    update_earth_moon_orbit_plot(orbit_plot, orbit_samples)
    plot_image.value = plot_to_png(orbit_plot)


time_slider = widgets.IntSlider(
    value=28 * 24,
    min=0,
    max=TOTAL_HOURS,
    step=TIME_STEP_HOURS,
    description="Hour:",
    continuous_update=True,
    layout=widgets.Layout(width="480px"),
)
hour_input = widgets.BoundedIntText(
    value=time_slider.value,
    min=time_slider.min,
    max=time_slider.max,
    step=TIME_STEP_HOURS,
    description="Hour:",
    layout=widgets.Layout(width="150px"),
)
back_button = widgets.Button(description="<", tooltip="Move back one hour", layout=widgets.Layout(width="40px"))
forward_button = widgets.Button(description=">", tooltip="Move forward one hour", layout=widgets.Layout(width="40px"))


def move_back(_button):
    time_slider.value = max(time_slider.min, time_slider.value - TIME_STEP_HOURS)


def move_forward(_button):
    time_slider.value = min(time_slider.max, time_slider.value + TIME_STEP_HOURS)


def sync_hour_input(change):
    hour_input.value = change["new"]


def sync_time_slider(change):
    time_slider.value = change["new"]


back_button.on_click(move_back)
forward_button.on_click(move_forward)
time_slider.observe(update_plot, names="value")
time_slider.observe(sync_hour_input, names="value")
hour_input.observe(sync_time_slider, names="value")
update_plot({"new": time_slider.value})

controls = widgets.HBox([back_button, time_slider, forward_button, hour_input])
display(controls, plot_image)

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x03 \x00\x00\x03 \x08\x06\x00\x00\x00\xdbp\x06h\x00\…

In [11]:
from datetime import datetime, timedelta, timezone
import importlib

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import earth_moon_model

importlib.reload(earth_moon_model)

from earth_moon_model import load_earth_moon_ephemeris
from earth_moon_orbit_plot import (
    create_earth_moon_orbit_plot,
    plot_to_png,
    update_earth_moon_orbit_plot,
)

EPHEMERIS_START = datetime(2026, 1, 1, tzinfo=timezone.utc)
EPHEMERIS_END = EPHEMERIS_START + timedelta(days=365)
SAMPLE_INTERVAL = timedelta(hours=1)
SUN_DISTANCE_COMPRESSION = 100

ephemeris_model = load_earth_moon_ephemeris(
    EPHEMERIS_START,
    EPHEMERIS_END,
    sample_interval=SAMPLE_INTERVAL,
    ephemeris="de432s",
)
ephemeris_3d_samples = ephemeris_model.moon_geocentric_samples
ephemeris_sun_positions = ephemeris_model.sun_geocentric_positions
ephemeris_samples = [
    (timestamp, (position[0], position[1])) for timestamp, position in ephemeris_3d_samples
]

ephemeris_plot = create_earth_moon_orbit_plot()
update_earth_moon_orbit_plot(ephemeris_plot, ephemeris_samples)
plt.close(ephemeris_plot.figure)

ephemeris_image = widgets.Image(
    value=plot_to_png(ephemeris_plot),
    format="png",
    layout=widgets.Layout(width="720px", height="720px"),
)
display(ephemeris_image)

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x03 \x00\x00\x03 \x08\x06\x00\x00\x00\xdbp\x06h\x00\…

In [13]:
import importlib

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

import earth_moon_orbit_plotly

importlib.reload(earth_moon_orbit_plotly)

from earth_moon_orbit_plotly import (
    create_earth_moon_orbit_plotly_3d,
    set_earth_moon_orbit_vertical_exaggeration,
    update_earth_moon_orbit_plotly_3d,
)

plotly_ephemeris_plot = create_earth_moon_orbit_plotly_3d()
plotly_ephemeris_plot.figure = go.FigureWidget(plotly_ephemeris_plot.figure)


def update_plotly_time(change):
    sample_index = change["new"]
    update_earth_moon_orbit_plotly_3d(
        plotly_ephemeris_plot,
        ephemeris_3d_samples[: sample_index + 1],
        sun_position=ephemeris_sun_positions[sample_index],
        sun_distance_compression=SUN_DISTANCE_COMPRESSION,
    )
    time_label.value = f"{ephemeris_3d_samples[sample_index][0]:%Y-%m-%d %H:%M UTC}"


def update_vertical_scale(change):
    set_earth_moon_orbit_vertical_exaggeration(plotly_ephemeris_plot, change["new"])


time_slider = widgets.IntSlider(
    value=len(ephemeris_3d_samples) - 1,
    min=0,
    max=len(ephemeris_3d_samples) - 1,
    step=1,
    description="Time:",
    continuous_update=True,
    layout=widgets.Layout(width="600px"),
)
time_label = widgets.Label()
vertical_scale = widgets.SelectionSlider(
    options=[("1x physical", 1), ("2x", 2), ("4x", 4), ("6x", 6), ("8x", 8)],
    value=1,
    description="Vertical scale:",
    continuous_update=False,
    layout=widgets.Layout(width="360px"),
)
play_button = widgets.Play(
    value=time_slider.value,
    min=time_slider.min,
    max=time_slider.max,
    step=time_slider.step,
    interval=75,
)
widgets.jslink((play_button, "value"), (time_slider, "value"))
time_slider.observe(update_plotly_time, names="value")
vertical_scale.observe(update_vertical_scale, names="value")
update_plotly_time({"new": time_slider.value})

display(
    widgets.VBox(
        [
            plotly_ephemeris_plot.figure,
            widgets.HBox([play_button, time_slider, time_label]),
            vertical_scale,
        ]
    )
)

    'data': [{'colorscale': [[0, '#dce8d8'], [1, '#dce8d8']],
              'hov…

In [14]:
import importlib

import earth_moon_threejs

importlib.reload(earth_moon_threejs)

from earth_moon_threejs import EarthMoonThreeJS

threejs_ephemeris = EarthMoonThreeJS(
    ephemeris_3d_samples,
    ephemeris_sun_positions,
    sun_distance_compression=100,
    vertical_exaggeration=1,
    moon_inclination_degrees=5.145,
    use_actual_moon_orbit=True,
    reference_frame="sun",
)
display(threejs_ephemeris)

In [10]:
import importlib

from IPython.display import display

import jupiter_galilean.galilean_moons_model as galilean_model

importlib.reload(galilean_model)

import jupiter_galilean.animate_galilean_moons as galilean_animation

importlib.reload(galilean_animation)

display(galilean_animation.create_jpl_widget())